# Auditable battery energy storage optimization

This notebook is a concise, executable companion to the formal methodology. It asks whether a 120 kWh / 40 kW behind-the-meter battery can reduce a commercial site's energy and demand bill after conversion losses, cycling cost, and a fair terminal state-of-charge condition are included.

**Evidence boundary:** all observations below are synthetic. The exercise demonstrates formulation, verification, and experimental discipline—not field performance.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
from bessopt.baselines import no_battery_dispatch
from bessopt.config import BatteryConfig, TariffConfig
from bessopt.data import generate_synthetic_data
from bessopt.metrics import compute_metrics, validate_dispatch
from bessopt.optimization import optimize_dispatch
from bessopt.plots import plot_dispatch

## 1. State assumptions before observing results

The battery has 90.25% round-trip efficiency, an 80% usable SOC window, a 3-hour nameplate duration, and a $0.02/kWh-throughput degradation proxy. A $12/kW demand charge applies to the study horizon. Export is disabled, and final SOC must equal initial SOC.

In [ ]:
battery = BatteryConfig()
tariff = TariffConfig()
data = generate_synthetic_data(days=30, seed=42)
data.describe().round(3)

## 2. Solve the sparse linear program

The no-storage comparator and optimizer see exactly the same time series and tariff.

In [ ]:
baseline = no_battery_dispatch(data, tariff)
solution = optimize_dispatch(data, battery, tariff)
dispatch = solution.dispatch

print(solution.solver_status)
print(f'Solve time: {solution.solve_time_seconds:.3f} s')
print(f'Maximum equality residual: {solution.max_equality_residual:.2e}')

## 3. Verify before interpreting

A plausible cost is not sufficient. The following checks independently reconstruct battery energy and site power balances and test for bound violations and opposing battery flows.

In [ ]:
checks = validate_dispatch(dispatch, battery)
pd.Series(checks, name='value').to_frame()

## 4. Report economic and technical outcomes together

In [ ]:
metrics = compute_metrics(dispatch, baseline, battery)
pd.Series(metrics, name='value').to_frame().round(3)

In [ ]:
plot_dispatch(dispatch, ROOT / 'figures')
from IPython.display import Image, display
display(Image(filename=ROOT / 'figures' / 'dispatch_week.png'))

## 5. Interpretation

The BESS can increase grid energy while lowering the bill: conversion losses require extra energy, but the controller shifts purchases away from expensive hours and suppresses the billing peak. This is an economic and flexibility result, not an energy-conservation result.

Run `python run_research.py` from the repository root for the capacity sweep, degradation sensitivity, 20-scenario Monte Carlo analysis, perfect/noisy-forecast MPC comparison, machine-readable provenance, and generated report.